# Análisis Detallado de Sensibilidad Emocional — Llama3, Mistral, Phi-3 y Nemotron3 (50 Iteraciones)
En este notebook se analizan los resultados de los experimentos realizados con 50 iteraciones para los modelos **Llama 3**, **Mistral**, **Phi-3** y **Nemotron3 33B**. El objetivo es evaluar la consistencia de los modelos y el impacto de diferentes emociones en sus respuestas.

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
from textblob import TextBlob
from scipy.stats import f_oneway
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

## 1. Carga y Unificación de Datasets
Cargamos únicamente los archivos CSV de **llama3**, **mistral**, **phi3** y **nemotron3** de la carpeta `dataset_experiemento_50iter/` y los combinamos en un solo DataFrame.

In [ ]:
ruta_datasets = "../dataset_experiemento_50iter/"

# Solo cargamos los modelos de interés
modelos_objetivo = ["llama3", "mistral", "phi3", "nemotron3_33b"]

dataframes = []
for modelo in modelos_objetivo:
    patron = os.path.join(ruta_datasets, f"sensibilidad_{modelo}_50iter.csv")
    archivos = glob.glob(patron)
    if not archivos:
        print(f"[ADVERTENCIA] No se encontró archivo para el modelo: {modelo}")
        continue
    for archivo in archivos:
        nombre_modelo = os.path.basename(archivo).replace("sensibilidad_", "").replace("_50iter.csv", "")
        df_temp = pd.read_csv(archivo)
        df_temp["model"] = nombre_modelo
        dataframes.append(df_temp)

df = pd.concat(dataframes, ignore_index=True)

# Renombrar columnas para consistencia si es necesario
df.rename(columns={
    "emocion": "emotion",
    "respuesta": "response",
    "longitud": "length"
}, inplace=True)

print(f"Total de registros cargados: {len(df)}")
print(f"Modelos cargados: {df['model'].unique().tolist()}")
df.head()

## 2. Feature Engineering
Calculamos métricas adicionales para profundizar en el análisis lingüístico y sentimental.

In [ ]:
def lexical_diversity(text):
    words = str(text).split()
    return len(set(words)) / len(words) if len(words) > 0 else 0

def sentence_length(text):
    sentences = nltk.sent_tokenize(str(text))
    if len(sentences) == 0:
        return 0
    return np.mean([len(s.split()) for s in sentences])

def get_sentiment(text):
    return TextBlob(str(text)).sentiment.polarity

def get_subjectivity(text):
    return TextBlob(str(text)).sentiment.subjectivity

def pronoun_usage(text):
    pronouns = ["yo", "tú", "te", "usted", "nosotros", "mí", "conmigo"]
    words = str(text).lower().split()
    return sum(1 for w in words if w in pronouns)

df["lexical_diversity"] = df["response"].apply(lexical_diversity)
df["sentence_length"] = df["response"].apply(sentence_length)
df["sentiment"] = df["response"].apply(get_sentiment)
df["subjectivity"] = df["response"].apply(get_subjectivity)
df["pronouns"] = df["response"].apply(pronoun_usage)

df.head()

## 3. Resumen Estadístico
Agrupamos por modelo y emoción para comparar los promedios.

In [ ]:
summary = df.groupby(["model", "emotion"]).agg({
    "length": ["mean", "std"],
    "sentiment": ["mean", "std"],
    "lexical_diversity": "mean",
    "sentence_length": "mean",
    "subjectivity": "mean",
    "pronouns": "mean"
}).reset_index()

summary

## 4. Variabilidad por Modelo
Medimos qué tan consistentes son los modelos en la longitud de sus respuestas.

In [ ]:
variability = df.groupby("model")["length"].std()
print("Desviación estándar de la longitud por modelo:")
print(variability)

## 5. Análisis de Varianza (ANOVA)
Evaluamos si las diferencias observadas entre emociones son estadísticamente significativas.

In [ ]:
def anova_effect(df, variable, model_name):
    subset = df[df["model"] == model_name]
    groups = [subset[subset["emotion"] == e][variable] for e in subset["emotion"].unique()]
    
    if len(groups) < 2:
        return np.nan, np.nan, np.nan
        
    f_stat, p_val = f_oneway(*groups)
    
    grand_mean = subset[variable].mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = sum((subset[variable] - grand_mean)**2)
    eta_sq = ss_between / ss_total if ss_total != 0 else 0
    
    return f_stat, p_val, eta_sq

for model in df["model"].unique():
    f, p, eta = anova_effect(df, "length", model)
    print(f"Modelo: {model} -> F={f:.3f}, p={p:.5f}, eta²={eta:.3f}")

## 6. Visualizaciones

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="emotion", y="length", hue="model")
plt.title("Distribución de Longitud de Respuesta por Emoción y Modelo (50 Iteraciones)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="emotion", y="sentiment", hue="model")
plt.title("Distribución de Sentimiento por Emoción y Modelo (50 Iteraciones)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="emotion", y="lexical_diversity", hue="model")
plt.title("Diversidad Léxica por Emoción y Modelo (50 Iteraciones)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="emotion", y="subjectivity", hue="model")
plt.title("Subjetividad por Emoción y Modelo (50 Iteraciones)")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
pivot_sentiment = df.groupby(["model", "emotion"])["sentiment"].mean().unstack()
plt.figure(figsize=(10, 5))
sns.heatmap(pivot_sentiment, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Sentimiento Promedio por Modelo y Emoción")
plt.tight_layout()
plt.show()

In [ ]:
pivot_length = df.groupby(["model", "emotion"])["length"].mean().unstack()
plt.figure(figsize=(10, 5))
sns.heatmap(pivot_length, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Longitud Promedio de Respuesta por Modelo y Emoción")
plt.tight_layout()
plt.show()